In [29]:
!mkdir cargacriolla

mkdir: cannot create directory ‘cargacriolla’: File exists


In [30]:
%%writefile cargacriolla/CargaCriolla.java
package cargacriolla;

import java.util.ArrayList;
import java.util.List;

public class CargaCriolla
{

    public static void main(String[] args)
    {

        if (args[0] == null || args[1] == null || args.length != 2)
        {
            System.out.println("error en parametros");
            return;
        }

        int trucksAmount = Integer.parseInt(args[0]);
        int travelsAmount = Integer.parseInt(args[1]);

        ShipmentsAdministrator shipmentsAdministrator = ShipmentsAdministrator.getShipmentsAdministrator(createTrucks(trucksAmount), createTravels(travelsAmount));

        shipmentsAdministrator.startAssigningTravels();

    }

    public static List<Truck> createTrucks(int trucksAmount){
        List<Truck> trucks = new ArrayList<>();

        for (int i = 1; i <= trucksAmount; i++)
        {
            trucks.add(new Truck(i));
        }
        return trucks;
    }

    public static List<Travel> createTravels(int travelsAmount){
        List<Plant> plants = List.of(new BsAsPlant(), new FernandezPlant(), new BsAsPlant());
        List<Travel> travels = new ArrayList<>();

        for(int i = 0; i<travelsAmount; i++ ){
            travels.add(new Travel(plants));
        }
        return travels;
    }
}

Overwriting cargacriolla/CargaCriolla.java


In [31]:
%%writefile cargacriolla/Plant.java
package cargacriolla;

import java.util.concurrent.Semaphore;
import java.util.concurrent.TimeUnit;

public abstract class Plant
{
    private final String name;
    private final String loadCommodity;
    private final String unloadCommodity;
    private final Semaphore loadingSemaphore = new Semaphore(Constants.MAX_LOADING_TRUCKS);
    private final Semaphore unloadingSemaphore = new Semaphore(Constants.MAX_UNLOADING_TRUCKS);

    public Plant(String name, String loadCommodity, String unloadCommodity)
    {
        this.name = name;
        this.loadCommodity = loadCommodity;
        this.unloadCommodity = unloadCommodity;
    }

    public void action(Truck truck) throws InterruptedException
    {
        if(truck.isLoaded()){
            this.unloadTruck(truck);
        }
        this.loadTruck(truck);
    }

    private void unloadTruck(Truck truck) throws InterruptedException
    {
        if(!this.unloadingSemaphore.tryAcquire(0, TimeUnit.SECONDS)){
            System.out.println(truck + " esperando para descargar " +this.unloadCommodity + " en "+ this.name);
            this.unloadingSemaphore.acquire();
        }
        System.out.println(truck +" descargando "+this.unloadCommodity +" en "+this.name);
        truck.unload();
        this.unloadingSemaphore.release();
    }

    private void loadTruck(Truck truck) throws InterruptedException
    {
        if(!this.loadingSemaphore.tryAcquire(0, TimeUnit.SECONDS)){
            System.out.println(truck + " esperando para cargar " +this.loadCommodity + " en "+ this.name);
            this.loadingSemaphore.acquire();
        }
        System.out.println(truck +" cargando "+this.loadCommodity +" en "+this.name);
        truck.load();
        this.loadingSemaphore.release();
    }

    public String getName()
    {
        return name;
    }
}


Overwriting cargacriolla/Plant.java


In [32]:
%%writefile cargacriolla/Travel.java
package cargacriolla;

import java.util.List;

public class Travel
{
    List<Plant> plants;

    public Travel(List<Plant> plants){
        this.plants = plants;
    }

    public Plant getNextStop(Plant currentPlant){
        int nextStopIndex = this.plants.indexOf(currentPlant) + 1;

        if(nextStopIndex < plants.size()){
            return plants.get(nextStopIndex);
        }

        return null;
    }

    public Plant getInitialStop(){
        return this.plants.get(0);
    }
}


Overwriting cargacriolla/Travel.java


In [33]:
%%writefile cargacriolla/Truck.java
package cargacriolla;

public class Truck implements Runnable
{

    private boolean isLoaded;

    int number;
    Plant currentPlant;
    Travel travel;

    Driver driver;

    public Truck(int number)
    {
        this.number = number;
        this.isLoaded = false;
    }

    public void assignTravel(Travel travel)
    {
        this.travel = travel;
        this.driver = new Driver();
        this.currentPlant = travel.getInitialStop();
    }

    public void goToNextStop() throws InterruptedException
    {
        System.out.println("Camion "+this.number+" sale de "+this.currentPlant.getName());
        int travelTime = Math.round(this.driver.getTravelTime()* Constants.SIMULATION_DELAY);
        Thread.sleep(travelTime);
        this.currentPlant = travel.getNextStop(currentPlant);
        System.out.format("Camion "+this.number+" llega a "+ currentPlant.getName()+ "(%.2f hs de viaje)%n",travelTime/(float)Constants.SIMULATION_DELAY);

    }

    public boolean isLoaded()
    {
        return this.isLoaded;
    }

    public void unload() throws InterruptedException
    {
        Thread.sleep(Constants.LOAD_OR_UNLOAD_TIME * Constants.SIMULATION_DELAY);
        this.isLoaded = false;
        System.out.println(this + " descargado");
    }

    public void load() throws InterruptedException
    {
        Thread.sleep(Constants.LOAD_OR_UNLOAD_TIME * Constants.SIMULATION_DELAY);
        System.out.println(this + " cargado");
        this.isLoaded = true;
    }

    @Override
    public void run()
    {
        System.out.println(this + " iniciando viaje");
        try{
            this.currentPlant.action(this);
            while (travel.getNextStop(this.currentPlant) != null){
                this.goToNextStop();
                this.currentPlant.action(this);
            }
            System.out.println(this + " finalizo viaje");
            this.travel = null;
        }catch (InterruptedException e){
            System.out.println("Error en hilo de camion "+this.number);
        }
    }

    public boolean isTraveling(){
        return travel != null;
    }

    @Override
    public String toString()
    {
        return "Camion " + number;
    }
}


Overwriting cargacriolla/Truck.java


In [34]:
%%writefile cargacriolla/ShipmentsAdministrator.java
package cargacriolla;

import java.time.Duration;
import java.util.List;
import java.util.Optional;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.ExecutorService;
import java.util.concurrent.Executors;
import java.util.concurrent.TimeUnit;

public class ShipmentsAdministrator
{
    private static ShipmentsAdministrator shipmentsAdministrator;

    List<Truck> trucks;
    List<Travel> travels;
    ExecutorService executor;
    CountDownLatch latch;

    public static ShipmentsAdministrator getShipmentsAdministrator(List<Truck> trucks, List<Travel> travels){
        if (shipmentsAdministrator == null){
            shipmentsAdministrator = new ShipmentsAdministrator(trucks, travels);
        }

        return shipmentsAdministrator;
    }

    private ShipmentsAdministrator(List<Truck> trucks, List<Travel> travels){
        this.trucks = trucks;
        this.travels = travels;
        this.executor = Executors.newFixedThreadPool(trucks.size());
        this.latch = new CountDownLatch(travels.size());

    }

    public void startAssigningTravels()
    {
        long initialTime = System.currentTimeMillis();
        while (!travels.isEmpty()){
            if(isAnyAvailableTruck())
            {
                Truck truck = findAvailableTruck();
                this.assignTravelToTruck(truck);
            }

        }
        System.out.println("No hay mas viajes");
        this.executor.shutdown();
        waitForTravelsToFinish();

        this.printTotalDays(initialTime, System.currentTimeMillis());
    }

    private void printTotalDays(long startTime, long finishTime){
        long hours = (finishTime - startTime)/Constants.SIMULATION_DELAY;
        Duration duration = Duration.ofHours(hours);
        long days = duration.toDays();
        System.out.println("Tiempo total: " + days +" dias");


    }

    private void waitForTravelsToFinish(){
        System.out.println("Esperando a que terminen los viajes en curso");
        try
        {
            latch.await();
        } catch (InterruptedException e)
        {
            System.out.println("Error mientras se finalizaban los viajes");
            throw new RuntimeException(e);
        }
        System.out.println("Todos los viajes finalizados");
    }

    private boolean isAnyTravelInCourse(){
        return this.trucks.stream().anyMatch(Truck::isTraveling);
    }

    private void assignTravelToTruck(Truck truck)
    {
        Travel travel = this.travels.remove(0);
        truck.assignTravel(travel);
        executor.submit(() ->
        {
            truck.run();
            latch.countDown();
        });
    }

    private boolean isAnyAvailableTruck(){
        return !this.trucks.stream().allMatch(Truck::isTraveling);
    }

    private Truck findAvailableTruck(){
        Optional<Truck> optTruck = trucks.stream().filter((t) -> !t.isTraveling()).findFirst();
        return optTruck.orElse(null);
    }
}


Overwriting cargacriolla/ShipmentsAdministrator.java


In [35]:
%%writefile cargacriolla/Driver.java
package cargacriolla;

import java.util.Random;

public class Driver
{
    private final static int MIN_TRAVEL_TIME = 18;
    private final static int MAX_TRAVEL_TIME = 24;

    private final float travelTime;

    public Driver(){
        Random random = new Random();
        this.travelTime = (random.nextFloat()*(MAX_TRAVEL_TIME-MIN_TRAVEL_TIME))+MIN_TRAVEL_TIME;
    }

    public float getTravelTime()
    {
        return travelTime;
    }
}


Overwriting cargacriolla/Driver.java


In [36]:
%%writefile cargacriolla/Constants.java
package cargacriolla;

public class Constants
{
    public static final int SIMULATION_DELAY = 100;
    public static final int LOAD_OR_UNLOAD_TIME = 2;
    public static final int MAX_LOADING_TRUCKS = 1;
    public static final int MAX_UNLOADING_TRUCKS = 1;
    public static final int PUMPS_AMOUNT = 2;
    public static final int FUEL_RECHARGE_TIME = 2;
}


Overwriting cargacriolla/Constants.java


In [37]:
%%writefile cargacriolla/FernandezPlant.java
package cargacriolla;

public class FernandezPlant extends Plant
{
    ChargeStation chargeStation;

    public FernandezPlant()
    {
        super("Fernandez", "Carbon", "Trigo");
        this.chargeStation = new ChargeStation();
    }

    @Override
    public void action(Truck truck) throws InterruptedException
    {
        super.action(truck);
        chargeStation.chargeTruck(truck);
    }
}


Overwriting cargacriolla/FernandezPlant.java


In [38]:
%%writefile cargacriolla/BsAsPlant.java
package cargacriolla;

public class BsAsPlant extends Plant
{
    public BsAsPlant()
    {
        super("Tapiales", "Trigo", "Carbon");
    }

    @Override
    public void action(Truck truck) throws InterruptedException
    {
        super.action(truck);
    }
}


Overwriting cargacriolla/BsAsPlant.java


In [39]:
%%writefile cargacriolla/ChargeStation.java
package cargacriolla;

import java.util.concurrent.Semaphore;
import java.util.concurrent.TimeUnit;

public class ChargeStation
{
    private final Semaphore pumpsSemaphore = new Semaphore(Constants.PUMPS_AMOUNT);

    public void chargeTruck(Truck truck) throws InterruptedException
    {
        if (!pumpsSemaphore.tryAcquire(0, TimeUnit.SECONDS)){
            System.out.println(truck+ " esperando para cargar combustible");
            this.pumpsSemaphore.acquire();
        }
        System.out.println(truck + " cargando combustible");
        Thread.sleep(Constants.FUEL_RECHARGE_TIME * Constants.SIMULATION_DELAY);
        this.pumpsSemaphore.release();
    }
}


Overwriting cargacriolla/ChargeStation.java


In [40]:
!find cargacriolla -name "*.java" > sources.txt
!javac @sources.txt

In [41]:
!java cargacriolla/CargaCriolla 4 10

Camion 1 iniciando viaje
Camion 2 iniciando viaje
Camion 3 iniciando viaje
Camion 4 iniciando viaje
Camion 3 esperando para cargar Trigo en Tapiales
Camion 2 esperando para cargar Trigo en Tapiales
Camion 4 esperando para cargar Trigo en Tapiales
Camion 1 cargando Trigo en Tapiales
Camion 1 cargado
Camion 3 cargando Trigo en Tapiales
Camion 1 sale de Tapiales
Camion 3 cargado
Camion 3 sale de Tapiales
Camion 2 cargando Trigo en Tapiales
Camion 2 cargado
Camion 2 sale de Tapiales
Camion 4 cargando Trigo en Tapiales
Camion 4 cargado
Camion 4 sale de Tapiales
Camion 3 llega a Fernandez(20.31 hs de viaje)
Camion 2 llega a Fernandez(19.37 hs de viaje)
Camion 2 esperando para descargar Trigo en Fernandez
Camion 3 descargando Trigo en Fernandez
Camion 1 llega a Fernandez(23.01 hs de viaje)
Camion 1 esperando para descargar Trigo en Fernandez
Camion 3 descargado
Camion 3 cargando Carbon en Fernandez
Camion 2 descargando Trigo en Fernandez
Camion 4 llega a Fernandez(19.77 hs de viaje)
Camion 4 

In [42]:
!ls cargacriolla/

BsAsPlant.class      Constants.java	   ShipmentsAdministrator.class
BsAsPlant.java	     Driver.class	   ShipmentsAdministrator.java
CargaCriolla.class   Driver.java	   Travel.class
CargaCriolla.java    FernandezPlant.class  Travel.java
ChargeStation.class  FernandezPlant.java   Truck.class
ChargeStation.java   Plant.class	   Truck.java
Constants.class      Plant.java
